# Large Model

Single models for the final pipeline, trained on a larger training subsample (100k texts instead of 50k, same validation split): an encoder with a larger backbone and more epochs, and the TF-IDF baseline. Ensembles and the chunking and perplexity experiments are added in a separate notebook. Training needs a GPU (GPU T4 x2 on Kaggle); a run that stops after an epoch resumes from its checkpoint.

In [ ]:
import sys
from pathlib import Path

# Make the repo importable: the local checkout, or a fresh clone on Kaggle/Colab. Then pin versions that must match.
REPO_URL = "https://github.com/BohdanZasymovych/ai-hw3-ai-text-detection.git"
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "setup_env.py").exists()), None)
if REPO is None:
    REPO = Path("/kaggle/working/hw3" if Path("/kaggle").exists() else "/content/hw3")
    if REPO.exists():
        !git -C {REPO} pull -q
    else:
        !git clone -q --depth 1 {REPO_URL} {REPO}
sys.path.insert(0, str(REPO))

import setup_env

setup_env.install_pins()

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer

from utils import (ARTIFACTS_DIR, DATA_DIR, ENV, LABEL_ORDER, LARGE_ENCODER, MODEL_COLORS, SEED, artifact_dir, cached,
                   class_report, compare_f1, compare_groups, ensure_data, ensure_dirs, find_artifact, get_split,
                   get_subsample, group_f1, length_band, macro_f1, plot_compare, plot_confusion, plot_group_f1,
                   submit_to_kaggle, write_submission)
from utils.encoder import head_tail_encode, predict_logits, train_encoder
from utils.tfidf import make_vectorizer

warnings.simplefilter("ignore", ConvergenceWarning)

ensure_dirs()
ensure_data()
split = get_split()  # fixed train/val split, made and cached on first use
gpu = f"{torch.cuda.device_count()}x {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "none"
print(f"env={ENV}  gpu={gpu}  data={DATA_DIR}  artifacts={ARTIFACTS_DIR}")

SUBMIT = False  # True: save the test predictions and upload the submission to Kaggle (5 uploads a day)

## Data

Load the fixed validation split, the test set, and a new training subsample of 100k texts from the train part (same rare-cell floor as the 50k one). It is saved under its own name, so the 50k comparison subsample is unchanged.

In [ ]:
N_TRAIN = 100_000
train_full = pd.read_csv(DATA_DIR / "train.csv")
subsample = get_subsample(n=N_TRAIN, name="subsample_100k")
train = train_full.iloc[subsample].reset_index(drop=True)
val = train_full.iloc[split["val"]].reset_index(drop=True)
del train_full
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
label_order = np.array(LABEL_ORDER)

pd.DataFrame({"texts": [len(train), len(val), len(test)]}, index=["train subsample", "validation", "test"])

## Tokenization

Head + tail truncation as in 03: a text longer than 510 tokens keeps its first 128 and last 382 tokens. The large backbone uses the same tokenizer as the base one. The token ids are cached.

In [ ]:
HEAD, TAIL = 128, 382  # 128 + 382 + <s> + </s> = 512 tokens
data_key = {"n_train": len(train), "n_val": len(val), "checksum": int(subsample.sum()), "v": 1}
tok_key = {**data_key, "tokenizer": LARGE_ENCODER, "head": HEAD, "tail": TAIL}

tokenizer = AutoTokenizer.from_pretrained(LARGE_ENCODER)
encoded = {name: cached(f"large_ids_{name}", lambda: head_tail_encode(df["text"], tokenizer, HEAD, TAIL),
                        key={**tok_key, "part": name})
           for name, df in [("train", train), ("val", val), ("test", test)]}
val["band"] = length_band(encoded["val"]["n_tokens"])

pd.DataFrame({
    name: {"texts": len(e["n_tokens"]), "median tokens": np.median(e["n_tokens"]),
           "truncated, %": (e["n_tokens"] > HEAD + TAIL).mean() * 100}
    for name, e in encoded.items()
}).T.style.format({"texts": "{:,.0f}", "median tokens": "{:,.0f}", "truncated, %": "{:.1f}%"})

## Training

Fine-tune `xlm-roberta-large` (24 layers, about 560M parameters) on the 100k subsample: AdamW, learning rate $1 \cdot 10^{-5}$ (lower than for base, because large encoders can collapse to one class with a higher one), 10% linear warmup and linear decay, 3 epochs, fp16. A batch of 16 texts is processed in 2 parts of 8 (gradient accumulation, to fit into memory; 4 texts per GPU on 2 GPUs), which gives the same updates as one batch of 16. The training state is saved after each epoch, so a stopped run resumes. Left: training loss averaged over every 50 steps. Right: validation macro-F1 after each epoch, for monitoring only; the reported model is the one after the last epoch.

In [ ]:
TRAIN = {"lr": 1e-5, "epochs": 3, "batch_size": 16, "grad_accum": 2, "warmup": 0.1, "weight_decay": 0.01, "seed": SEED}
KEY = {**tok_key, "backbone": LARGE_ENCODER, **TRAIN, "v": 1}
y_train = train["label"].map({label: i for i, label in enumerate(LABEL_ORDER)}).to_numpy()

model_dir = artifact_dir("encoder_large", key=KEY)
history = cached("encoder_large_history", lambda: train_encoder(
    LARGE_ENCODER, encoded["train"]["ids"], y_train, model_dir, val_ids=encoded["val"]["ids"],
    val_labels=val["label"].to_numpy(), checkpoint="encoder_large_ckpt", key=KEY, **TRAIN), key=KEY, kind="json")

loss, per_epoch = pd.DataFrame(history["loss"]), pd.DataFrame(history["val"])
fig, (ax_loss, ax_f1) = plt.subplots(1, 2, figsize=(11, 3), gridspec_kw={"width_ratios": [2, 1]})
ax_loss.plot(loss["step"], loss["loss"], color="#4C78A8")
for step in per_epoch["step"].iloc[:-1]:
    ax_loss.axvline(step, color="black", ls="--", lw=1)  # end of an epoch
ax_loss.set_xlabel("training step")
ax_loss.set_ylabel("cross-entropy loss")
ax_f1.plot(per_epoch["epoch"], per_epoch["macro_f1"], marker="o", color="#4C78A8")
for epoch, f1 in zip(per_epoch["epoch"], per_epoch["macro_f1"]):
    ax_f1.annotate(f"{f1:.3f}", (epoch, f1), textcoords="offset points", xytext=(0, 6), ha="center")
ax_f1.set_xticks(per_epoch["epoch"])
ax_f1.set_xlabel("epoch")
ax_f1.set_ylabel("validation macro-F1")
plt.tight_layout()
plt.show()

## Evaluation

Predict the validation set with the large encoder (logits cached) and show the per-class scores.

In [ ]:
val_logits = cached("encoder_large_val_logits", lambda: predict_logits(model_dir, encoded["val"]["ids"]), key=KEY, kind="npy")
val_pred = label_order[val_logits.argmax(axis=1)]
cached("encoder_large_val_metrics", lambda: {"macro_f1": macro_f1(val["label"], val_pred)}, key=KEY, kind="json")
class_report(val["label"], val_pred).style.format({"precision": "{:.3f}", "recall": "{:.3f}", "f1": "{:.3f}", "support": "{:,.0f}"})

Confusion matrix. Colours show the share of each true class; the counts are in brackets.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
plot_confusion(val["label"], val_pred, ax)
plt.show()

Macro-F1 per language and domain. The dashed line is the overall validation macro-F1.

In [ ]:
by_group = group_f1(val, val_pred, ["lang", "domain"])
fig, ax = plt.subplots(figsize=(7, 4))
plot_group_f1(by_group, macro_f1(val["label"], val_pred), ax)
plt.tight_layout()
plt.show()

## TF-IDF

The best baseline configuration from 02 (word 1-2-grams + character 3-5-grams, class-balanced logistic regression, C = 3), refitted on the 100k subsample. The vectorizers are fitted on the training texts only. Only the fitted model and its scores are cached: the feature matrices of 100k texts would take several GB.

In [ ]:
TFIDF = {"features": "word+char", "model": "LogisticRegression", "C": 3.0}  # best configuration in 02
tfidf_key = {**data_key, **TFIDF, "seed": SEED}


def fit_tfidf():
    vectorizers = [make_vectorizer(kind) for kind in TFIDF["features"].split("+")]
    X_train = sp.hstack([v.fit_transform(train["text"]) for v in vectorizers], format="csr")
    model = LogisticRegression(C=TFIDF["C"], class_weight="balanced", max_iter=200, random_state=SEED)
    model.fit(X_train, train["label"])
    del X_train
    order = [list(model.classes_).index(label) for label in LABEL_ORDER]
    scores = {name: model.decision_function(sp.hstack([v.transform(df["text"]) for v in vectorizers], format="csr"))[:, order]
              for name, df in [("val", val), ("test", test)]}
    return {"vectorizers": vectorizers, "model": model, **scores}


tfidf = cached("tfidf100k", fit_tfidf, key=tfidf_key)
for name in ["val", "test"]:
    cached(f"tfidf100k_{name}_scores", lambda: tfidf[name], key=tfidf_key, kind="npy")  # light copies for later notebooks
tfidf_pred = label_order[tfidf["val"].argmax(axis=1)]
class_report(val["label"], tfidf_pred).style.format({"precision": "{:.3f}", "recall": "{:.3f}", "f1": "{:.3f}", "support": "{:,.0f}"})

## Comparison with Baseline

F1 per class and macro-F1 of the single models on the same validation rows: the TF-IDF baseline on 50k (02) and 100k texts, and the base encoder on 50k (03) and the large one on 100k.

In [ ]:
preds = {
    "baseline": label_order[np.load(find_artifact("baseline_val_scores", "npy")).argmax(axis=1)],
    "baseline 100k": tfidf_pred,
    "encoder": label_order[np.load(find_artifact("encoder_val_logits", "npy")).argmax(axis=1)],
    "encoder large": val_pred,
}
compare_f1(val["label"], preds).style.background_gradient(cmap="Blues", axis=None).format("{:.3f}")

Macro-F1 by text length (left) and by language and domain (right).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [2, 3]})
plot_compare(compare_groups(val, preds, "band"), axes[0], MODEL_COLORS)
plot_compare(compare_groups(val, preds, ["lang", "domain"]), axes[1], MODEL_COLORS)
plt.tight_layout()
plt.show()

## Submission

Test logits of the large encoder are always predicted and cached, so the final notebook needs no GPU for them. Only when `SUBMIT` is True, save a valid submission file of the large encoder and upload it to Kaggle (cached, so a rerun does not spend another daily submission).

In [ ]:
test_logits = cached("encoder_large_test_logits", lambda: predict_logits(model_dir, encoded["test"]["ids"]), key=KEY, kind="npy")
test_pred = label_order[test_logits.argmax(axis=1)]

if SUBMIT:
    path = write_submission(test["id"], test_pred, sample_submission, "large")
    message = f"encoder large: {LARGE_ENCODER} {N_TRAIN // 1000}k texts, {TRAIN['epochs']} epochs, val macro-F1 {macro_f1(val['label'], val_pred):.3f}"
    cached("large_upload", lambda: submit_to_kaggle(path, message), key=KEY, kind="json")
    display(pd.Series(test_pred).value_counts(normalize=True).reindex(LABEL_ORDER).mul(100).round(1)
            .to_frame("share of test predictions, %").T)
else:
    print("SUBMIT is False: upload skipped")

## Conclusions

_To be written after the run._